# Malilangwe Wildlife Detector — WAID Fine-Tuning

This notebook fine-tunes YOLOv11 on the WAID aerial wildlife dataset.

**Before running:**
1. `Runtime → Change runtime type → T4 GPU`
2. Configure the settings cell below
3. `Runtime → Run all`

**Hitting rate limits?** Set `EPOCHS = 30` and `BATCH = 8` below (~50 min per session).  
Download `last.pt` at the end of each session and re-upload next time to resume.

## ⚙️ Settings — Edit These Before Running

In [ ]:
# ── How many epochs to train this session ──────────────────────────────────
# First time:   30  (fast, ~50 min, good for testing)
# Full run:     100 (best results, ~2-3 hrs — only if you have time)
EPOCHS = 30

# ── Batch size ─────────────────────────────────────────────────────────────
# 16 = faster but uses more memory (may crash on free T4)
# 8  = safer for free Colab
BATCH = 8

# ── Resume from a previous session? ───────────────────────────────────────
# False = start fresh from pretrained YOLOv11
# True  = upload last.pt from your previous session when prompted
RESUME = False

print(f'Settings: EPOCHS={EPOCHS}, BATCH={BATCH}, RESUME={RESUME}')

## 1. Check GPU

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode != 0:
    raise RuntimeError('No GPU found. Go to Runtime → Change runtime type → T4 GPU, then re-run.')
print(result.stdout)

## 2. Clone Repo & Install Dependencies

In [ ]:
import os

REPO_URL = 'https://github.com/Tadiwa-M/wildlife-detector-malilangwe.git'
REPO_DIR = '/content/wildlife-detector-malilangwe'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

os.chdir(REPO_DIR)
print('Working directory:', os.getcwd())

In [ ]:
!pip install -q -r requirements.txt
print('Dependencies installed.')

## 3. Upload Previous Checkpoint (Resume only)

**Skip this cell if `RESUME = False`.**  
If resuming, upload your `last.pt` from a previous session when prompted.

In [ ]:
CHECKPOINT_PATH = None

if RESUME:
    from google.colab import files as colab_files
    from pathlib import Path

    print('Upload your last.pt checkpoint file...')
    uploaded = colab_files.upload()

    if not uploaded:
        raise RuntimeError('No file uploaded. Upload last.pt to resume, or set RESUME=False.')

    uploaded_name = list(uploaded.keys())[0]
    dest = Path('runs/train/waid_yolo11n/weights')
    dest.mkdir(parents=True, exist_ok=True)
    os.rename(uploaded_name, dest / 'last.pt')
    CHECKPOINT_PATH = str(dest / 'last.pt')
    print(f'Checkpoint saved to {CHECKPOINT_PATH}')
else:
    print('RESUME=False — starting fresh from pretrained weights.')

## 4. Download WAID Dataset (Images + Labels)

In [ ]:
WAID_DIR = '/content/wildlife-detector-malilangwe/WAID'

if not os.path.exists(os.path.join(WAID_DIR, 'WAID', 'images')):
    print('Downloading WAID dataset (~2 GB, takes a few minutes)...')
    !git clone --depth=1 https://github.com/xiaohuicui/WAID.git {WAID_DIR}/WAID_repo
    !cp -r {WAID_DIR}/WAID_repo/WAID/images {WAID_DIR}/WAID/images
    print('WAID images downloaded.')
else:
    print('WAID images already present, skipping download.')

## 5. Validate Dataset & Generate YAML

In [ ]:
import sys
sys.path.insert(0, REPO_DIR)

from src.config import load_config
from src.data.dataset import validate_dataset, get_class_distribution, generate_dataset_yaml
from src.utils.logging_setup import setup_logging

cfg = load_config()
setup_logging(cfg)

stats = validate_dataset(cfg)
print(f"\nTotal images : {stats['total_images']:,}")
print(f"Total labels : {stats['total_labels']:,}")

print('\nClass distribution (train):')
dist = get_class_distribution(cfg, split='train')
for name, count in sorted(dist.items(), key=lambda x: -x[1]):
    print(f'  {name:<12s} {count:>8,}')

dataset_yaml = generate_dataset_yaml(cfg)
print('\nDataset YAML:', dataset_yaml)

## 6. Train

Using your settings from the top of the notebook.  
Progress prints every epoch — watch mAP50 climb.

In [ ]:
from ultralytics import YOLO

train_cfg = cfg.training
det_cfg = cfg.detection

if RESUME and CHECKPOINT_PATH:
    print(f'Resuming from {CHECKPOINT_PATH}')
    model = YOLO(CHECKPOINT_PATH)
    resume_flag = True
else:
    print(f'Starting fresh from pretrained {det_cfg.model_variant}.pt')
    model = YOLO(f"{det_cfg.model_variant}.pt")
    resume_flag = False

results = model.train(
    data=str(dataset_yaml),
    epochs=EPOCHS,
    batch=BATCH,
    imgsz=int(train_cfg.image_size),
    optimizer=str(train_cfg.optimizer),
    lr0=float(train_cfg.learning_rate),
    weight_decay=float(train_cfg.weight_decay),
    patience=int(train_cfg.patience),
    device=0,
    hsv_h=float(train_cfg.augmentation.hsv_h),
    hsv_s=float(train_cfg.augmentation.hsv_s),
    hsv_v=float(train_cfg.augmentation.hsv_v),
    flipud=float(train_cfg.augmentation.flipud),
    fliplr=float(train_cfg.augmentation.fliplr),
    mosaic=float(train_cfg.augmentation.mosaic),
    mixup=float(train_cfg.augmentation.mixup),
    scale=float(train_cfg.augmentation.scale),
    project='runs/train',
    name='waid_yolo11n',
    exist_ok=True,
    resume=resume_flag,
)

print('\nTraining complete!')
print('Best mAP50:', results.results_dict.get('metrics/mAP50(B)', 'N/A'))

## 7. Evaluate on Test Split

In [ ]:
from pathlib import Path

best_weights = Path('runs/train/waid_yolo11n/weights/best.pt')

if best_weights.exists():
    eval_model = YOLO(str(best_weights))
    metrics = eval_model.val(
        data=str(dataset_yaml),
        split='test',
        conf=float(det_cfg.confidence_threshold),
        iou=float(det_cfg.iou_threshold),
        imgsz=int(det_cfg.image_size),
        device=0,
        plots=True,
    )
    print('\n' + '='*50)
    print(f'  mAP@50:    {metrics.box.map50:.4f}')
    print(f'  mAP@50-95: {metrics.box.map:.4f}')
    print(f'  Precision:  {metrics.box.mp:.4f}')
    print(f'  Recall:     {metrics.box.mr:.4f}')
    print('='*50)
else:
    print(f'best.pt not found — training may not have finished yet.')

## 8. Download Weights

Downloads both `best.pt` (overall best) and `last.pt` (most recent epoch).  
- **Keep training next session?** → re-upload `last.pt` and set `RESUME = True`  
- **Done training?** → put `best.pt` in your local `weights/` folder

In [ ]:
from google.colab import files
import shutil
from pathlib import Path

weights_dir = Path('runs/train/waid_yolo11n/weights')
best = weights_dir / 'best.pt'
last = weights_dir / 'last.pt'

if best.exists():
    shutil.copy(best, '/content/waid_best.pt')
    files.download('/content/waid_best.pt')
    print('Downloading best.pt ...')

if last.exists():
    shutil.copy(last, '/content/waid_last.pt')
    files.download('/content/waid_last.pt')
    print('Downloading last.pt (use this to resume next session) ...')

print('\nDone!')
print('  → To continue training: set RESUME=True and re-upload waid_last.pt')
print('  → To use locally:       put waid_best.pt in your weights/ folder')

## 9. Quick Test on a Custom Image (Optional)

Upload any aerial image to see what the model detects.

In [ ]:
from google.colab import files as colab_files
from IPython.display import Image, display
import glob

print('Upload an aerial image to test...')
uploaded = colab_files.upload()

if uploaded:
    img_path = list(uploaded.keys())[0]
    test_model = YOLO('runs/train/waid_yolo11n/weights/best.pt')
    results = test_model.predict(
        source=img_path,
        conf=0.15,
        save=True,
        project='/content/test_output',
        name='result',
        exist_ok=True,
    )

    for r in results:
        print(f'Detections: {len(r.boxes)}')
        for box in r.boxes:
            cls = int(box.cls[0])
            conf = float(box.conf[0])
            print(f'  {r.names[cls]}: {conf:.0%}')

    saved = glob.glob('/content/test_output/result/*.jpg') + glob.glob('/content/test_output/result/*.png')
    if saved:
        display(Image(saved[0]))